# LangGraph Level 2C — Architectural Patterns

Patterns covered:
1. **Orchestrator-Worker** — a planner generates tasks, workers execute them in parallel, synthesizer combines results
2. **Evaluator-Optimizer** — generate output → evaluate it → loop until sufficient quality

> These patterns are the foundation of multi-agent systems. Level 3 extends them with the `Send` API and Supervisor pattern.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from typing import Literal, Annotated, TypedDict
import operator
from IPython.display import Image, display

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from pydantic import BaseModel

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.types import Command

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

---
## 1. Orchestrator-Worker Pattern

**What is it?** A three-layer architecture:
- **Orchestrator**: decomposes the objective into concrete tasks
- **Workers**: each worker executes a specific task in a specialized way
- **Synthesizer**: combines all results into a coherent final output

```
Input → [Orchestrator] → task_list
                       → [Worker A] → result_a
                       → [Worker B] → result_b  → [Synthesizer] → Output
                       → [Worker C] → result_c
```

**Difference from Routing:** Routing = 1 branch. Orchestrator = multiple coordinated workers.

**Difference from Level 3 Supervisor:** Here the orchestrator generates a fixed plan. The Supervisor dynamically decides which agent to call at each step.

In [ ]:
class ResearchPlan(BaseModel):
    tasks: list[str]
    focus: str

class ResearchState(MessagesState):
    research_question: str
    tasks: list[str]
    results: Annotated[list[str], operator.add]  # reducer: append (does not replace)
    synthesis: str

planner_llm = llm.with_structured_output(ResearchPlan)

In [ ]:
def orchestrator(state: ResearchState) -> ResearchState:
    """Planner: decomposes the question into 3 research subtasks."""
    plan = planner_llm.invoke([
        SystemMessage(content=(
            "You are a research orchestrator. "
            "Given a topic, create exactly 3 distinct and complementary research tasks. "
            "Each task should be a specific question or aspect of the topic."
        )),
        HumanMessage(content=f"Topic: {state['research_question']}")
    ])
    print(f"[Orchestrator] Plan generated: {len(plan.tasks)} tasks")
    for i, task in enumerate(plan.tasks, 1):
        print(f"  Task {i}: {task}")
    return {"tasks": plan.tasks}

def worker_1(state: ResearchState) -> ResearchState:
    """Worker specialized in technical aspects."""
    if not state["tasks"]:
        return {"results": ["No task assigned"]}
    task = state["tasks"][0]
    response = llm.invoke([
        SystemMessage(content="You are a technical researcher. Respond in 2-3 concise sentences."),
        HumanMessage(content=f"Research: {task}")
    ])
    print(f"[Worker 1] Completed task 1")
    return {"results": [f"[Technical] {response.content}"]}

def worker_2(state: ResearchState) -> ResearchState:
    """Worker specialized in practical use cases."""
    if len(state["tasks"]) < 2:
        return {"results": ["No task assigned"]}
    task = state["tasks"][1]
    response = llm.invoke([
        SystemMessage(content="You are a practical applications expert. Respond with concrete examples in 2-3 sentences."),
        HumanMessage(content=f"Research: {task}")
    ])
    print(f"[Worker 2] Completed task 2")
    return {"results": [f"[Practical] {response.content}"]}

def worker_3(state: ResearchState) -> ResearchState:
    """Worker specialized in trends and the future."""
    if len(state["tasks"]) < 3:
        return {"results": ["No task assigned"]}
    task = state["tasks"][2]
    response = llm.invoke([
        SystemMessage(content="You are a trends analyst. Respond with a forward-looking perspective in 2-3 sentences."),
        HumanMessage(content=f"Research: {task}")
    ])
    print(f"[Worker 3] Completed task 3")
    return {"results": [f"[Trends] {response.content}"]}

def synthesizer(state: ResearchState) -> ResearchState:
    """Combines all results into a coherent analysis."""
    all_results = "\n\n".join(state["results"])
    response = llm.invoke([
        SystemMessage(content=(
            "You are an expert synthesizer. "
            "Combine these research results into a cohesive 3-4 paragraph analysis. "
            "Connect the dots and extract the key conclusions."
        )),
        HumanMessage(content=f"Original question: {state['research_question']}\n\nResults:\n{all_results}")
    ])
    print(f"[Synthesizer] Synthesis completed")
    return {"synthesis": response.content, "messages": [AIMessage(content=response.content)]}

In [ ]:
# Orchestrator-Worker graph: orchestrator → 3 workers (parallel) → synthesizer
builder = StateGraph(ResearchState)
builder.add_node("orchestrator", orchestrator)
builder.add_node("worker_1", worker_1)
builder.add_node("worker_2", worker_2)
builder.add_node("worker_3", worker_3)
builder.add_node("synthesizer", synthesizer)

builder.add_edge(START, "orchestrator")

# All 3 workers start in parallel after the orchestrator
builder.add_edge("orchestrator", "worker_1")
builder.add_edge("orchestrator", "worker_2")
builder.add_edge("orchestrator", "worker_3")

# The synthesizer waits for all workers to finish
builder.add_edge("worker_1", "synthesizer")
builder.add_edge("worker_2", "synthesizer")
builder.add_edge("worker_3", "synthesizer")

builder.add_edge("synthesizer", END)

orchestrator_graph = builder.compile()
display(Image(orchestrator_graph.get_graph().draw_mermaid_png()))

In [ ]:
result = orchestrator_graph.invoke({
    "research_question": "What is the impact of LangGraph on the development of multi-agent systems in 2025?"
})

print("\n=== FINAL SYNTHESIS ===")
print(result["synthesis"])
print(f"\nIndividual results: {len(result['results'])} workers")

---
## 2. Evaluator-Optimizer Pattern

**What is it?** An iterative improvement loop:
1. **Generator**: produces an output (text, code, plan)
2. **Evaluator**: scores the output with specific criteria
3. **Router**: if quality is sufficient → finish. If not → return to generator with feedback

```
Input → [Generator] → output
            ↑              ↓
         feedback    [Evaluator] → score ≥ threshold → END
            ↑              ↓
            └──── [Router] ────┘ score < threshold → loop
```

**When to use it:** When you need guaranteed quality (code that compiles, text that meets criteria, responses without hallucinations).

In [ ]:
class EvalScore(BaseModel):
    score: int  # 1-10
    feedback: str
    passed: bool

class ContentState(MessagesState):
    objective: str
    draft: str
    score: int
    feedback: str
    iteration: int
    history: Annotated[list[str], operator.add]  # saves all drafts

evaluator_llm = llm.with_structured_output(EvalScore)

QUALITY_THRESHOLD = 7
MAX_ITERATIONS = 3

In [ ]:
def generator(state: ContentState) -> ContentState:
    """Generates or improves content based on previous feedback."""
    iteration = state.get("iteration", 0) + 1
    feedback = state.get("feedback", "")
    previous_draft = state.get("draft", "")
    
    if previous_draft and feedback:
        # Second iteration onwards: improve with feedback
        prompt = (
            f"Objective: {state['objective']}\n\n"
            f"Previous draft (score {state.get('score', 0)}/10):\n{previous_draft}\n\n"
            f"Evaluator feedback: {feedback}\n\n"
            f"Improve the draft by applying the feedback."
        )
    else:
        # First iteration: generate from scratch
        prompt = f"Write: {state['objective']}"
    
    response = llm.invoke([
        SystemMessage(content="You are an expert writer. Produce high-quality content."),
        HumanMessage(content=prompt)
    ])
    
    print(f"[Generator] Iteration {iteration} — generating draft")
    return {
        "draft": response.content,
        "iteration": iteration,
        "history": [f"[iter-{iteration}] {response.content[:100]}..."]
    }

def evaluator(state: ContentState) -> ContentState:
    """Evaluates the draft with specific criteria and gives a 1-10 score."""
    eval_result = evaluator_llm.invoke([
        SystemMessage(content=(
            "Evaluate this content with strict criteria:\n"
            "- Clarity: is it easy to understand?\n"
            "- Conciseness: does it get to the point without filler?\n"
            "- Technical accuracy: are the facts correct?\n"
            "- Action: does it have a clear call-to-action or conclusion?\n"
            "Score 1-10 (7+ = pass). Be demanding."
        )),
        HumanMessage(content=f"Objective: {state['objective']}\n\nContent:\n{state['draft']}")
    ])
    
    print(f"[Evaluator] Score: {eval_result.score}/10 | Passed: {eval_result.passed}")
    print(f"  Feedback: {eval_result.feedback[:150]}")
    return {"score": eval_result.score, "feedback": eval_result.feedback}

def route_quality(state: ContentState) -> Command[Literal["generator", "__end__"]]:
    """Decides if the output is good enough or needs another iteration."""
    score = state.get("score", 0)
    iteration = state.get("iteration", 0)
    
    if score >= QUALITY_THRESHOLD or iteration >= MAX_ITERATIONS:
        reason = "quality reached" if score >= QUALITY_THRESHOLD else f"limit of {MAX_ITERATIONS} iterations"
        print(f"[Router] → DONE ({reason}, score={score})")
        final_msg = AIMessage(content=state["draft"])
        return Command(update={"messages": [final_msg]}, goto=END)
    else:
        print(f"[Router] → RETRY (score={score} < {QUALITY_THRESHOLD}, iter={iteration})")
        return Command(goto="generator")

In [ ]:
builder = StateGraph(ContentState)
builder.add_node("generator", generator)
builder.add_node("evaluator", evaluator)
builder.add_node("route_quality", route_quality)

builder.add_edge(START, "generator")
builder.add_edge("generator", "evaluator")
builder.add_edge("evaluator", "route_quality")
# route_quality uses Command to decide: → generator (retry) or → END

eval_graph = builder.compile()
display(Image(eval_graph.get_graph().draw_mermaid_png()))

In [ ]:
print("=== Evaluator-Optimizer in action ===")
print(f"Quality threshold: {QUALITY_THRESHOLD}/10 | Max iterations: {MAX_ITERATIONS}")
print()

result = eval_graph.invoke({
    "objective": (
        "Explain in 3 paragraphs why AI Engineers should learn LangGraph. "
        "It should be persuasive, technical, and end with a concrete call to action."
    )
})

print(f"\n=== FINAL RESULT (score={result['score']}/10, iterations={result['iteration']}) ===")
print(result["draft"])

if len(result.get("history", [])) > 1:
    print(f"\n=== DRAFT HISTORY ({len(result['history'])} versions) ===")
    for h in result["history"]:
        print(f"  {h}")

---
## Comparison: Orchestrator-Worker vs Evaluator-Optimizer

| | Orchestrator-Worker | Evaluator-Optimizer |
|---|---|---|
| **Structure** | Horizontal (parallelism) | Vertical (iteration) |
| **Flow** | Plan → multiple workers in parallel | Generate → evaluate → loop |
| **Goal** | Complete diverse tasks simultaneously | Improve quality of a single output |
| **Key state** | `results: Annotated[list, operator.add]` (merge) | `score`, `feedback`, `iteration` |
| **Use case** | Research, data pipeline, multi-angle analysis | Text generation, code, any output with a quality criterion |
| **Risk** | Worker synchronization | Infinite loops (→ `MAX_ITERATIONS`) |

---
## Takeaways — N2C

### Orchestrator-Worker
- The `Annotated[list, operator.add]` in the state is key: it allows multiple workers to write to the same list without overwriting each other
- Workers can run in parallel by adding multiple `add_edge("orchestrator", "worker_X")`
- The synthesizer receives the accumulated state from all workers

### Evaluator-Optimizer
- `MAX_ITERATIONS` is mandatory to avoid infinite loops
- The evaluator's feedback is passed to the generator in the next iteration
- `Command(goto=END)` terminates the loop when the threshold is reached
- Saving the draft history allows auditing the improvement

**Next level:** `N3_multiagent.ipynb` — Supervisor pattern · Swarm · Send API · Long-term memory